## Mapping UMLS vaccine terms to VO

In [13]:
import requests
from typing import List, Sequence, Tuple, Optional
import pandas as pd
import pickle
from tqdm import tqdm
import concurrent.futures
import time
from threading import Lock
from functools import wraps

Rationale: UMLS has two concepts (CUIs) related to vaccines: 
1. Vaccines (C0042210)
2. Vaccine [APC] (C5399710)

The goal would be to: 
1. Base CUI [X]
2. Get all atoms (AUIs) of the two Vaccine concepts (occurences within source vocabularies) - Limited to english language. This yields 36 AUIs
3. Get descendents for all atoms (AUIs) extracted in previous step
4. For all unique concepts (CUIs) extracted in previous step, extract related atoms (RN/RO/RQ/SY/SIB)
5. For each newly identified CUI, extract relations again (with a depth limit) to find nested vaccine products/formulations

### UMLS API [https://documentation.uts.nlm.nih.gov/rest/home.html]
eg: https://uts-ws.nlm.nih.gov/rest/content/current/AUI/A0131114/descendants?apiKey=0c4d7175-6688-4c45-9a13-4c64d7354673

In [2]:
apikey = '0c4d7175-6688-4c45-9a13-4c64d7354673'
uri = 'https://uts-ws.nlm.nih.gov/rest'
UMLS_S_DESCENDANTS_URL = f'{uri}/content/current/source/umls/descendants'
UMLS_A_DESCENDANTS_URL = f'{uri}/content/current/AUI/{{aui}}/descendants'
UMLS_ATOM_URL = f'{uri}/content/current/CUI/{{cui}}/atoms'

# Get descendants of vaccine concepts: 

In [ ]:
# !! get comparison of vocabs in umls vs athena !! 
 # BASIC STATS (vocab,  # concepts)


In [3]:
# Retrieve all atoms for UMLS vaccines (C0042210, C5399710)
#https://documentation.uts.nlm.nih.gov/rest/atoms/
def get_umls_atoms(cui: str, apiKey : str, lang = "ENG", pageNumber = 1 ) -> List[dict]:
    all_atoms = []
    UMLS_ATOM_URL = f'{uri}/content/current/CUI/{{cui}}/atoms'
    while True: 
        response = requests.get(UMLS_ATOM_URL.format(cui=cui), params={'apiKey': apiKey, 'language': lang, 'pageNumber': pageNumber})
        response.encoding = 'utf-8'
        if response.status_code != 200:
            break
        atoms = response.json().get('result', {})
        if not atoms:
            break
        all_atoms.extend(atoms)
        pageNumber += 1
    return all_atoms

In [4]:
umls_vaccine_atoms = get_umls_atoms('C0042210', apikey)
umls_vaccine_atoms += get_umls_atoms('C5399710', apikey)

In [7]:
vac_atom_df = pd.DataFrame(umls_vaccine_atoms)
vac_atom_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents
0,Atom,A24665667,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,HC,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
1,Atom,A22722839,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,ATC,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,VACCINES,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
2,Atom,A32282011,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE
3,Atom,A0131113,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,LCH,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,NONE,NONE,NONE,NONE,NONE,NONE
4,Atom,A23870698,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,LCH_NW,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Vaccines,NONE,NONE,NONE,NONE,NONE,NONE
5,Atom,A1397511,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,AOD,DE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,vaccines,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
6,Atom,A0806394,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,SNMI,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Therapeutic vaccine, NOS",https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...
7,Atom,A19725017,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,HL7V3.0,VS,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,VaccineEntityType,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE
8,Atom,A12101066,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,VANDF,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,VACCINES,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE
9,Atom,A18618187,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,CHV,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,therapeutic vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE


In [8]:
# Retrieve all descendants for a given UMLS Atom
# https://documentation.uts.nlm.nih.gov/rest/atoms/ancestors-and-descendants/index.html
def get_umls_descendants(aui: str, apiKey: str, lang: str = "ENG", pageNumber: int = 1) -> List[dict]:
    all_descendants = []
    UMLS_DESCENDANT_URL = f'{uri}/content/current/AUI/{{aui}}/descendants'
    while True:
        response = requests.get(UMLS_DESCENDANT_URL.format(aui=aui), params={'apiKey': apiKey, 'language': lang, 'pageNumber': pageNumber})
        response.encoding = 'utf-8'
        if response.status_code != 200:
            break
        descendants = response.json().get('result', {})
        if not descendants:
            break
        all_descendants.extend(descendants)
        pageNumber += 1
    return all_descendants

In [9]:
umls_vaccine_descendants = []
for atom in umls_vaccine_atoms:
    ui = atom.get('ui')
    if ui:
        descendants = get_umls_descendants(ui, apikey)
        umls_vaccine_descendants.extend(descendants)

In [ ]:
#load umls_vaccine_descendants into a pandas dataframe
vac_df = pd.DataFrame(umls_vaccine_descendants)
vac_df.head()
vac_df.to_pickle('results/umls_vaccine_descendants.pkl')

In [4]:
vac_df = pd.read_pickle('results/umls_vaccine_descendants.pkl')

In [6]:
# new column: codeClean where all string after the last '/' is extracted from the code column
vac_df['codeClean'] = vac_df['code'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
vac_df['conceptClean'] = vac_df['concept'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
vac_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents,codeClean,conceptClean
0,Atom,A24665364,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria and Tetanus Toxoids and Acellular P...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MTHU003345,C0535644
1,Atom,A24665543,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria and Tetanus Toxoids Adsorbed (DT),https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MTHU003344,C2984509
2,Atom,A24665606,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Rotavirus Vaccine Live,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MTHU003368,C0795661
3,Atom,A24665740,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria and Tetanus Toxoids and Acellular P...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MTHU003347,C3526551
4,Atom,A27873227,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Meningococcal Group B Vaccine,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MTHU005110,C3859491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3313,Atom,A28941361,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Live Attenuated Salmonella Typhi Vaccine,NONE,NONE,NONE,NONE,NONE,NONE,N0000183910,C3257579
3314,Atom,A28941362,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Inactivated Human Papillomavirus Vaccine,NONE,NONE,NONE,NONE,NONE,NONE,N0000183891,C3257549
3315,Atom,A28941373,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Live Attenuated Varicella Zoster Virus Vaccine,NONE,NONE,NONE,NONE,NONE,NONE,N0000183911,C3257586
3316,Atom,A28941400,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Inactivated Corynebacterium Diphtheriae Vaccine,NONE,NONE,NONE,NONE,NONE,NONE,N0000183887,C3257545


In [4]:
vac_df['rootSource'].value_counts()

rootSource
NCI            1494
MEDCIN          651
SNOMEDCT_US     468
RCD             323
ATC             131
MSH             120
CSP              35
USPMG            32
SNMI             31
MED-RT           30
AOD               3
Name: count, dtype: int64

#### Test Case: Missing Concept A36802459 - C5941869
> 0.3 ML SARS-CoV-2 (COVID-19) vaccine, mRNA-BNT162b2 OMICRON (KP.2) 0.1 MG/ML Prefilled Syringe [Comirnaty 2024-2025]

In [7]:
# check if 'A36802459' is in 'ui' column 
vac_df[vac_df['ui'] == 'A36802459']

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents,codeClean,conceptClean


### Workaround: Try to explode concept column to get atoms again. 

In [11]:
cui_list = vac_df['conceptClean'].unique().tolist()

In [12]:
len(cui_list)

2333

In [18]:
class RateLimiter:
    def __init__(self, max_calls_per_second=5):
        self.max_calls_per_second = max_calls_per_second
        self.min_interval = 1.0 / max_calls_per_second
        self.last_called = 0
        self.lock = Lock()
    
    def wait(self):
        with self.lock:
            elapsed = time.time() - self.last_called
            left_to_wait = self.min_interval - elapsed
            if left_to_wait > 0:
                time.sleep(left_to_wait)
            self.last_called = time.time()

rate_limiter = RateLimiter(max_calls_per_second=10)  # Adjust based on API limits

def get_umls_atoms_with_rate_limit(cui: str, apiKey: str, lang: str = "ENG") -> List[dict]:
    """Rate-limited version of get_umls_atoms"""
    rate_limiter.wait()
    return get_umls_atoms(cui, apiKey, lang)

def get_all_atoms_threaded(cui_list: List[str], apikey: str, max_workers: int = 10):
    """Use ThreadPoolExecutor for concurrent requests"""
    exploded_atoms = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_cui = {
            executor.submit(get_umls_atoms_with_rate_limit, cui, apikey): cui 
            for cui in cui_list
        }
        
        # Track progress
        with tqdm(total=len(cui_list), desc="Fetching UMLS atoms") as pbar:
            for future in concurrent.futures.as_completed(future_to_cui):
                cui = future_to_cui[future]
                try:
                    atoms = future.result()
                    exploded_atoms.extend(atoms)
                    pbar.set_postfix({"Current CUI": cui, "Atoms": len(atoms)})
                except Exception as e:
                    print(f"Error for CUI {cui}: {e}")
                pbar.update(1)
    
    return exploded_atoms

In [19]:
exploded_atoms = get_all_atoms_threaded(cui_list, apikey, max_workers=10)

Fetching UMLS atoms: 100%|██████████| 2333/2333 [03:54<00:00,  9.97it/s, Current CUI=C5889747, Atoms=4] 


In [20]:
len(exploded_atoms)

9630

In [ ]:
exploded_df = pd.DataFrame(exploded_atoms)
exploded_df.head()
exploded_df.to_pickle('results/umls_exploded_atoms.pkl')

In [5]:
exploded_df = pd.read_pickle('results/umls_exploded_atoms.pkl')

In [8]:
exploded_df['codeClean'] = exploded_df['code'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
exploded_df['conceptClean'] = exploded_df['concept'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else x)
exploded_df

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents,codeClean,conceptClean
0,Atom,A29944320,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,PDQ,AB,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Td adsorbed,NONE,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,CDR0000561778,C2984509
1,Atom,A29935138,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,PDQ,AB,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,DT,NONE,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,CDR0000561778,C2984509
2,Atom,A20236080,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,NCI,SY,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria and Tetanus Toxoids Adsorbed USP (F...,NONE,NONE,NONE,NONE,NONE,NONE,C91718,C2984509
3,Atom,A24665543,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,USPMG,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria and Tetanus Toxoids Adsorbed (DT),https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,MTHU003344,C2984509
4,Atom,A20577328,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MTH,PN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria Toxoid/Tetanus Toxoid Vaccine Adsorbed,NONE,NONE,NONE,NONE,NONE,NONE,NOCODE,C2984509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9625,Atom,A28937986,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,FN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Inactivated Corynebacterium Diphtheriae Vaccin...,NONE,NONE,NONE,NONE,NONE,NONE,N0000183887,C3257545
9626,Atom,A36855672,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,PT,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,COVID-19 mRNA Vaccine,NONE,NONE,NONE,NONE,NONE,NONE,N0000194097,C5889747
9627,Atom,A36855674,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MED-RT,FN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,COVID-19 mRNA Vaccine [EPC],NONE,NONE,NONE,NONE,NONE,NONE,N0000194097,C5889747
9628,Atom,A36855803,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MTH,PN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,COVID-19 mRNA Vaccine [EPC],NONE,NONE,NONE,NONE,NONE,NONE,NOCODE,C5889747


In [7]:
exploded_df['rootSource'].value_counts()

rootSource
NCI              2733
MEDCIN           1435
PDQ              1364
SNOMEDCT_US      1153
MSH               787
DRUGBANK          303
CHV               250
RCD               231
MTH               220
CVX               144
ATC               139
MMSL               85
NDDF               80
SNMI               72
VANDF              64
MED-RT             63
CSP                62
RXNORM             56
HL7V2.5            50
HL7V3.0            49
USPMG              33
LCH_NW             33
SNM                32
GS                 27
LNC                26
RCDAE              25
MTHSPL             16
MDR                15
LCH                12
SNOMEDCT_VET       11
MEDLINEPLUS        11
ICPC2P              9
AOD                 8
MMX                 6
HCPCS               4
ICPC2ICD10ENG       2
ICD10AM             2
ICD10PCS            2
CPT                 2
ICD10               1
CCPSS               1
ICD10CM             1
ICD10AMAE           1
CDT                 1
MTHICD9             1

In [9]:
exploded_df[exploded_df['ui'] == 'A36802459']

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents,codeClean,conceptClean


In [8]:
exploded_df[exploded_df['rootSource'] == 'RXNORM']

,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents,codeClean,conceptClean
52,Atom,A24180157,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,meningococcal group B vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,1593128,C3859491
328,Atom,A31765460,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"typhoid Vi polysaccharide vaccine, S typhi Ty2...",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,807219,C0078221
472,Atom,A20970167,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"tetanus toxoid vaccine, inactivated",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,798306,C0305062
563,Atom,A31730731,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,pertussis vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,8080,C0031237
595,Atom,A20966660,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"diphtheria toxoid vaccine, inactivated",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,798304,C0012551
966,Atom,A31631956,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,varicella-zoster virus vaccine live (Oka-Merck...,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,1292422,C3464497
1000,Atom,A32463564,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"SARS-CoV-2 (COVID-19) vaccine, mRNA spike protein",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,2468231,C5424007
1005,Atom,A32658502,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"SARS-COV-2 (COVID-19) vaccine, vector non-repl...",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,2479831,C5442843
1068,Atom,A20963127,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,IN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"acellular pertussis vaccine, inactivated",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,798302,C0982332
1148,Atom,A24778441,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,RXNORM,PSN,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,HAVRIX adult vaccine 1440 UNT in 1 ML Prefille...,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,798479,C1322283


In [12]:
exploded_df[exploded_df['rootSource'] == 'MSH']


,classType,ui,sourceDescriptor,sourceConcept,concept,suppressible,obsolete,rootSource,termType,code,language,name,ancestors,descendants,attributes,relations,children,parents,codeClean,conceptClean
7,Atom,A33264326,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria Tetanus acellular Pertussis Vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE,D022681,C0535644
8,Atom,A33268810,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,PM,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccine, Diphtheria-Tetanus-acellular Pertussis",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D022681,C0535644
11,Atom,A1956799,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,PM,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria Tetanus acellular Pertussis Vaccines,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D022681,C0535644
14,Atom,A1956801,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,MH,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Diphtheria-Tetanus-acellular Pertussis Vaccines,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D022681,C0535644
15,Atom,A26632198,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,DTaP Vaccines,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D022681,C0535644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8966,Atom,A26601295,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Live Attenuated Mumps Vaccine,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,NONE,D009108,C3850126
9260,Atom,A0131103,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,PM,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccine, Paratyphoid",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D014436,C0030529
9261,Atom,A26671780,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,ET,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,"Vaccines, Paratyphoid",NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D014436,C0030529
9262,Atom,A0097811,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,false,false,MSH,PM,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,ENG,Paratyphoid Vaccines,NONE,NONE,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,https://uts-ws.nlm.nih.gov/rest/content/2025AA...,NONE,NONE,D014436,C0030529


In [7]:
# concatonate exploded_df and vac_df 
combined_df = pd.concat([exploded_df, vac_df], ignore_index=True)
# remove duplicates based on 'ui' column
combined_df = combined_df.drop_duplicates(subset='ui', keep='first')
combined_df['rootSource'].value_counts()

rootSource
NCI              2733
MEDCIN           1435
PDQ              1364
SNOMEDCT_US      1153
MSH               787
RCD               332
DRUGBANK          303
CHV               250
MTH               220
CVX               144
ATC               139
MMSL               85
NDDF               80
SNMI               72
VANDF              64
MED-RT             63
CSP                62
RXNORM             56
HL7V2.5            50
HL7V3.0            49
USPMG              33
LCH_NW             33
SNM                32
GS                 27
LNC                26
RCDAE              25
MTHSPL             16
MDR                15
LCH                12
SNOMEDCT_VET       11
MEDLINEPLUS        11
ICPC2P              9
AOD                 8
MMX                 6
HCPCS               4
ICPC2ICD10ENG       2
ICD10AM             2
ICD10PCS            2
CPT                 2
ICD10               1
CCPSS               1
ICD10CM             1
ICD10AMAE           1
CDT                 1
MTHICD9             1

In [17]:
# Unique number of conceptClean column
combined_df['conceptClean'].nunique()

2333

In [10]:
combined_df

NameError: name 'combined_df' is not defined

In [8]:
# read csv into dataframe
athena_df = pd.read_csv('results/Athena search[vaccine_valid_drug].csv', sep='\t')
athena_df.head()

,Id,Code,Name,Standard Class,Domain,Vocab,Validity,Concept
0,28493,D000068756,Valsartan,Main Heading,Drug,MeSH,Valid,Non-standard
1,28686,C576297,ponceau 4R,Suppl Concept,Drug,MeSH,Valid,Non-standard
2,32766,OMOP4873977,Middle East Respiratory Syndrome heterologous ...,Ingredient,Drug,RxNorm Extension,Valid,Standard
3,509104,347699,meningococcal group A polysaccharide 0.1 MG/ML...,Clinical Drug,Drug,RxNorm,Valid,Standard
4,514012,314724,meningococcal polysaccharide vaccine group W-135,Ingredient,Drug,RxNorm,Valid,Standard


In [11]:
athena_df

,Id,Code,Name,Standard Class,Domain,Vocab,Validity,Concept
0,28493,D000068756,Valsartan,Main Heading,Drug,MeSH,Valid,Non-standard
1,28686,C576297,ponceau 4R,Suppl Concept,Drug,MeSH,Valid,Non-standard
2,32766,OMOP4873977,Middle East Respiratory Syndrome heterologous ...,Ingredient,Drug,RxNorm Extension,Valid,Standard
3,509104,347699,meningococcal group A polysaccharide 0.1 MG/ML...,Clinical Drug,Drug,RxNorm,Valid,Standard
4,514012,314724,meningococcal polysaccharide vaccine group W-135,Ingredient,Drug,RxNorm,Valid,Standard
...,...,...,...,...,...,...,...,...
18437,46367132,428740015,influenza vaccine 45ug/.5mL INTRAMUSCULAR INJE...,9-digit NDC,Drug,NDC,Valid,Non-standard
18438,46367232,437420579,"adenosinum triphosphoricum dinatrum, apiolum, ...",9-digit NDC,Drug,NDC,Valid,Non-standard
18439,46367270,449110164,"american cheese, blue cheese, brie cheese, che...",9-digit NDC,Drug,NDC,Valid,Non-standard
18440,46368290,581600903,influenza virus vaccine 15ug/.5mL INTRAMUSCULA...,9-digit NDC,Drug,NDC,Valid,Non-standard


In [14]:
athena_df['Vocab'].value_counts()

Vocab
RxNorm Extension     9156
RxNorm               2020
SPL                  1759
NDC                   862
dm+d                  840
AMT                   672
CPT4                  468
NDFRT                 333
SNOMED Veterinary     311
GCN_SEQNO             280
Nebraska Lexicon      255
SNOMED                250
MeSH                  202
CVX                   198
VANDF                 162
OMOP Invest Drug      153
BDPM                   88
DPD                    81
CIEL                   70
JMDC                   57
ATC                    52
Multum                 45
NCCD                   23
HCPCS                  21
EphMRA ATC             18
KDC                    13
CTD                    13
ICD10PCS               12
Read                   11
CO-CONNECT              8
HemOnc                  4
VA Class                3
GGR                     1
EDI                     1
Name: count, dtype: int64

In [9]:
# show the valuecounts of the 'Vocab' column (athena_df) and 'rootSource' column (combined_df) side by side
vocab_counts = athena_df['Vocab'].value_counts()
root_source_counts = combined_df['rootSource'].value_counts()
comparison_df = pd.DataFrame({'Athena': vocab_counts,
'UMLS': root_source_counts}).fillna(0).astype(int)
comparison_df

,Athena,UMLS
AMT,672,0
AOD,0,8
ATC,52,139
BDPM,88,0
BI,0,1
...,...,...
USP,0,1
USPMG,0,33
VA Class,3,0
VANDF,162,64


In [10]:
comparison_df

,Athena,UMLS
AMT,672,0
AOD,0,8
ATC,52,139
BDPM,88,0
BI,0,1
...,...,...
USP,0,1
USPMG,0,33
VA Class,3,0
VANDF,162,64


### ToDo: 
1. Add function to get related concepts (https://documentation.uts.nlm.nih.gov/rest/relations/)
2. Normalize vocab sources acros Athena and UMLS
3. Check vocabulary code to identify missing concepts from each source